In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/titanic/train.csv
/kaggle/input/competitions/titanic/test.csv
/kaggle/input/competitions/titanic/gender_submission.csv


# Pipelineの使用
今回、データに特徴量を追加、前処理、モデル
作成までをPipelineで一括管理することを目指す

In [2]:
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier

# データの読み込みと分割

In [3]:
train_csv = pd.read_csv("/kaggle/input/competitions/titanic/train.csv").set_index("PassengerId")
test_csv = pd.read_csv("/kaggle/input/competitions/titanic/test.csv").set_index("PassengerId")

y = train_csv.Survived
X = train_csv.drop("Survived", axis=1)

X_train,X_valid,y_train,y_valid = train_test_split(X,y,test_size=0.2,random_state=0)

# カスタム特徴量を作るトランスフォーマー

In [4]:
class TitanicFeatureEngineering(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass #paramatoeはない

    def fit(self,X,y=None):
        return self #今回は特徴量作成に統計データ使わないのでfit要らない

    def transform(self,X): #ここから新特徴量の作成
        X_new = X.copy()

        #Sex_Pclassの作成
        X_new["Sex_Pclass"] = (
            X_new["Sex"].astype(str) + "_" +
            X_new["Pclass"].astype(str)
        )

        #logFareの作成
        X_new["logFare"] = np.log1p(X_new["Fare"])

        #len_famの作成
        family = X_new["Parch"] + X_new["SibSp"]
        X_new["len_fam"] = family.apply(
            lambda x: (
                "alone" if x == 0 else
                "basic" if 1<=x<=3 else
                "laege" 
            )
        )

        return X_new
        

# Pipelineの構築

In [5]:
#数値列に対する前処理
numerical_transformer = Pipeline(
    steps = [
        ("imputer", SimpleImputer(strategy="median"))
    ]
)

#カテゴリ列に対する前処理
categorical_transformer = Pipeline(
    steps = [
        ("imputer",SimpleImputer(strategy="most_frequent")),
        ("onehot",OneHotEncoder(handle_unknown="ignore",sparse_output = False))
    ]
)

#使用する列を明示
num_cols = ["Age","SibSp","Parch","Fare","logFare"]
cat_cols = ["Sex","Embarked","Sex_Pclass","len_fam"]

#前処理の結合
preprocessor = ColumnTransformer(
    transformers=[
        ("num",numerical_transformer,num_cols),
        ("cat",categorical_transformer,cat_cols)
    ]
)

#すべての工程をPipelineにまとめる
my_pipeline = Pipeline(
    steps = [
        ("feature_engineering",TitanicFeatureEngineering()),
        ("preprocessor",preprocessor),
        ("model",XGBClassifier(
            n_estimators = 100,
            learning_rate = 0.05,
            max_depth = 5,
            random_state = 10
        ))
    ]
)


# 学習と予測

In [6]:
my_pipeline.fit(X_train,y_train)

val_accuracy = my_pipeline.score(X_valid,y_valid)
print(val_accuracy)

preds = my_pipeline.predict(test_csv)

output = pd.DataFrame({"PassengerId":test_csv.index,"Survived":preds})
output.to_csv("submission.csv",index=False)

0.8491620111731844
